# LG Aimers 9기 — 제구 성공 확률 예측 `model_v3`

이 노트북은 **이미 최종 채택·제출했던 M2_contam 구조만 baseline으로 사용**합니다.

다시 돌리지 않는 것:
- M0 → M1 → M2 순차 비교
- reliability 단독 ablation
- game_type 제거
- M2_current vs M2_contam 재비교
- R/F hard expert
- V2 LightGBM/ensemble

이번 V3 핵심:
1. **Sample weighting**
2. **Baseline + residual**

Seed 정책:
- screening: seed 42 한 번
- 실험 bagging: **5 seeds**
- 최종 제출: **10 seeds만 한 번**

기존 기준점:
- Val2024 M2_contam 5-seed: **801.146099**
- Public anchor: **840**

## 실행 흐름

```text
데이터 로드
→ 제출 모델(M2_contam) 피처 생성
→ weighting 2024 single-seed screening
→ 상위 weighting 2023 stress
→ weighting 승자 5-seed
→ baseline + honest residual 2024
→ residual 2023 stress
→ residual 승자 5-seed
→ 최종 구조 확정
→ 그때만 10-seed full train
```

Public 점수로 보정값을 역산하지 않고, test 행 간 집계도 사용하지 않습니다.

In [1]:
from pathlib import Path
import gc, random, time, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
SCREEN_SEED = 42
EXPERIMENT_SEEDS = [11, 22, 33, 44, 55]
FINAL_SEEDS = [11, 22, 33, 44, 55, 66, 77, 88, 99, 111]

np.random.seed(SEED)
random.seed(SEED)

DATA_DIR = Path("/Users/joyeeun/Desktop/LG Aimers 9기/open/data")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_PATH = DATA_DIR / "sample_submission.csv"

TARGET = "control_success"
ID_COL = "row_id"

REFERENCE_VAL2024_5SEED = 801.146099
REFERENCE_PUBLIC = 840.0

print("screen seed:", SCREEN_SEED)
print("experiment seeds:", EXPERIMENT_SEEDS)
print("final seeds:", FINAL_SEEDS)

screen seed: 42
experiment seeds: [11, 22, 33, 44, 55]
final seeds: [11, 22, 33, 44, 55, 66, 77, 88, 99, 111]


In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

assert TARGET in train.columns
assert TARGET not in test.columns
assert ID_COL in train.columns and ID_COL in test.columns

print("train:", train.shape)
print("test :", test.shape)

display(
    train.groupby(["season", "game_type"])[TARGET]
    .agg(["mean", "count"])
)

train: (1475092, 49)
test : (5, 48)


mean   count
season game_type                  
2019   F          0.689250   25786
       R          0.549490  211627
2020   F          0.587774   23213
       R          0.526925  220874
2021   F          0.703840   25861
       R          0.512763  221227
2022   F          0.708749   30448
       R          0.503691  217024
2023   F          0.472904   25686
       R          0.503118  219839
2024   F          0.459280   30010
       R          0.489707  223497

## 1. 평가 함수

In [3]:
def competition_like_score(y_true, pred):
    y_true = np.asarray(y_true, dtype=float)
    pred = np.clip(np.asarray(pred, dtype=float), 1e-6, 1 - 1e-6)
    bs = np.mean((pred - y_true) ** 2)
    r = float(np.mean(y_true))
    null_bs = r * (1 - r)
    raw = 100000 * (1 - bs / null_bs)
    return {
        "brier": float(bs),
        "score_like": float(max(0.0, raw)),
        "pred_mean": float(pred.mean()),
        "actual_rate": r,
        "mean_gap": float(pred.mean() - r),
        "pred_std": float(pred.std()),
    }

def evaluate_by_game_type(pred_df, full_train):
    tmp = pred_df.merge(
        full_train[[ID_COL, "game_type"]].drop_duplicates(ID_COL),
        on=ID_COL,
        how="left",
    )
    rows = []
    for subset, part in [
        ("ALL", tmp),
        ("R", tmp[tmp["game_type"].astype(str) == "R"]),
        ("F", tmp[tmp["game_type"].astype(str) == "F"]),
    ]:
        if len(part):
            rows.append({
                "subset": subset,
                "n": len(part),
                **competition_like_score(part["y_true"], part["pred"]),
            })
    return pd.DataFrame(rows)

## 2. 제출 모델 피처 파이프라인

최종 제출했던 구조만 유지합니다.

```text
공통 전처리
+ reliability
+ leakage-safe platoon EB
+ game_type
+ contamination 3피처
```

In [4]:
CATEGORICAL_CANDIDATES = [
    "game_dayofweek", "top_bottom", "game_type", "base_state",
    "pitcher_hand", "batter_hand", "pitcher_team_id", "batter_team_id",
]

RATE_COLS = [
    c for c in train.columns
    if c.startswith("asof_") and c.endswith("_rate")
]

RELIABILITY_K = 200.0
PLATOON_K = 200.0

CONTAM_FEATURES = [
    "fe_pitcher_futures_share",
    "fe_batter_futures_share",
    "fe_pitcher_prior_n_log",
]

BASE_DROP_MODEL = {
    TARGET, ID_COL, "pitcher_id", "batter_id",
    "pitcher_eb", "pitcher_handmatch_eb", "p_n", "ph_n",
}

In [5]:
def fit_preprocess_state(df_fit):
    state = {
        "target_prior": float(df_fit[TARGET].mean()),
        "rate_median": {},
    }
    for c in RATE_COLS:
        if c in df_fit.columns:
            med = df_fit[c].median()
            if pd.isna(med):
                med = state["target_prior"] if "success_rate" in c else 0.0
            state["rate_median"][c] = float(med)
    return state

def apply_common_preprocess(df, state):
    out = df.copy()

    for c in CATEGORICAL_CANDIDATES:
        if c in out.columns:
            out[c] = (
                out[c].astype("object")
                .where(out[c].notna(), "__MISSING__")
                .astype(str)
            )

    important_missing_cols = [
        "asof_pitcher_success_rate",
        "asof_pitcher_prev1_game_success_rate",
        "asof_pitcher_prev3_game_success_rate",
        "asof_pitcher_prev5_game_success_rate",
        "asof_pitcher_pitchmix_n",
        "asof_pitcher_fastball_rate",
        "asof_pitcher_breaking_rate",
        "asof_pitcher_offspeed_rate",
        "asof_batter_success_rate",
    ]

    for c in important_missing_cols:
        if c in out.columns:
            out[f"{c}__missing"] = out[c].isna().astype("int8")

    career = "asof_pitcher_success_rate"
    if career in out.columns:
        fallback = state["rate_median"].get(career, state["target_prior"])
        out[career] = out[career].fillna(fallback)
        for c in [
            "asof_pitcher_prev1_game_success_rate",
            "asof_pitcher_prev3_game_success_rate",
            "asof_pitcher_prev5_game_success_rate",
        ]:
            if c in out.columns:
                out[c] = out[c].fillna(out[career])

    if "asof_batter_success_rate" in out.columns:
        out["asof_batter_success_rate"] = out["asof_batter_success_rate"].fillna(
            state["rate_median"].get("asof_batter_success_rate", state["target_prior"])
        )

    for c in RATE_COLS:
        if c in out.columns:
            out[c] = out[c].fillna(state["rate_median"].get(c, 0.0))

    return out

def add_reliability_features(df):
    out = df.copy()
    for c in ["asof_pitcher_n", "asof_batter_n", "asof_pitcher_pitchmix_n"]:
        if c in out.columns:
            x = pd.to_numeric(out[c], errors="coerce").fillna(0).clip(lower=0)
            out[f"log1p__{c}"] = np.log1p(x)
            out[f"reliability__{c}"] = x / (x + RELIABILITY_K)
    return out

In [6]:
def fit_platoon_lookup(df_fit, k=PLATOON_K):
    prior = float(df_fit[TARGET].mean())

    gp = (
        df_fit.groupby("pitcher_id", dropna=False)[TARGET]
        .agg(["sum", "count"]).reset_index()
        .rename(columns={"sum":"p_success", "count":"p_n"})
    )
    gp["pitcher_eb"] = (gp["p_success"] + k * prior) / (gp["p_n"] + k)

    gh = (
        df_fit.groupby(["pitcher_id", "batter_hand"], dropna=False)[TARGET]
        .agg(["sum", "count"]).reset_index()
        .rename(columns={"sum":"ph_success", "count":"ph_n"})
    )
    gh["pitcher_handmatch_eb"] = (
        gh["ph_success"] + k * prior
    ) / (gh["ph_n"] + k)

    return {
        "prior": prior,
        "pitcher": gp[["pitcher_id", "p_n", "pitcher_eb"]],
        "hand": gh[["pitcher_id", "batter_hand", "ph_n", "pitcher_handmatch_eb"]],
    }

def apply_platoon_lookup(df, lookup, k=PLATOON_K):
    out = df.copy()
    out = out.merge(lookup["pitcher"], on="pitcher_id", how="left")
    out = out.merge(lookup["hand"], on=["pitcher_id", "batter_hand"], how="left")

    prior = lookup["prior"]
    out["p_n"] = out["p_n"].fillna(0)
    out["ph_n"] = out["ph_n"].fillna(0)
    out["pitcher_eb"] = out["pitcher_eb"].fillna(prior)
    out["pitcher_handmatch_eb"] = out["pitcher_handmatch_eb"].fillna(out["pitcher_eb"])
    out["platoon_split_eb"] = out["pitcher_handmatch_eb"] - out["pitcher_eb"]
    out["platoon_n_reliability"] = out["ph_n"] / (out["ph_n"] + k)
    return out

def add_temporal_platoon_features(df_train, k=PLATOON_K):
    out = df_train.copy()
    generated = pd.DataFrame(
        index=out.index,
        columns=["platoon_split_eb", "platoon_n_reliability"],
        dtype=float,
    )

    for season in sorted(out["season"].dropna().unique()):
        idx = out.index[out["season"] == season]
        history = out.loc[out["season"] < season]

        if len(history) == 0:
            generated.loc[idx, :] = 0.0
            continue

        enc = apply_platoon_lookup(
            out.loc[idx],
            fit_platoon_lookup(history, k=k),
            k=k,
        )
        generated.loc[idx, "platoon_split_eb"] = enc["platoon_split_eb"].values
        generated.loc[idx, "platoon_n_reliability"] = enc["platoon_n_reliability"].values

    out["platoon_split_eb"] = generated["platoon_split_eb"].fillna(0.0)
    out["platoon_n_reliability"] = generated["platoon_n_reliability"].fillna(0.0)
    return out

In [7]:
def fit_futures_lookup(df_history):
    hist = df_history.copy()
    is_f = hist["game_type"].astype(str).eq("F").astype(float)

    tmp = pd.DataFrame({
        "pitcher_id": hist["pitcher_id"].values,
        "batter_id": hist["batter_id"].values,
        "_is_f": is_f.values,
    })

    p = tmp.groupby("pitcher_id", dropna=False)["_is_f"].agg(["mean", "count"])
    b = tmp.groupby("batter_id", dropna=False)["_is_f"].mean()

    return {
        "pitcher_futures_share": p["mean"].to_dict(),
        "pitcher_prior_n": p["count"].to_dict(),
        "batter_futures_share": b.to_dict(),
    }

def apply_futures_lookup(df, lookup):
    out = df.copy()
    out["fe_pitcher_futures_share"] = out["pitcher_id"].map(
        lookup["pitcher_futures_share"]
    ).astype(float)
    out["fe_batter_futures_share"] = out["batter_id"].map(
        lookup["batter_futures_share"]
    ).astype(float)
    pn = out["pitcher_id"].map(lookup["pitcher_prior_n"]).astype(float)
    out["fe_pitcher_prior_n_log"] = np.log1p(pn)
    return out

def add_temporal_futures_features(df_train_raw):
    out = df_train_raw.copy()
    generated = pd.DataFrame(index=out.index, columns=CONTAM_FEATURES, dtype=float)

    for season in sorted(out["season"].dropna().unique()):
        idx = out.index[out["season"] == season]
        history = out.loc[out["season"] < season]
        if len(history) == 0:
            continue
        enc = apply_futures_lookup(out.loc[idx], fit_futures_lookup(history))
        generated.loc[idx, CONTAM_FEATURES] = enc[CONTAM_FEATURES].values

    for c in CONTAM_FEATURES:
        out[c] = generated[c]
    return out

In [8]:
def get_model_features(df):
    return [c for c in df.columns if c not in BASE_DROP_MODEL]

def get_cat_feature_indices(features):
    cat_names = [c for c in CATEGORICAL_CANDIDATES if c in features]
    return [features.index(c) for c in cat_names], cat_names

def build_submitted_fold(full_train, val_year):
    """최종 제출했던 M2_contam 구조만 생성."""
    tr_raw = full_train.loc[full_train["season"] < val_year].copy()
    va_raw = full_train.loc[full_train["season"] == val_year].copy()

    if len(tr_raw) == 0 or len(va_raw) == 0:
        raise ValueError(f"val_year={val_year}: rows 부족")

    state = fit_preprocess_state(tr_raw)
    tr = add_reliability_features(apply_common_preprocess(tr_raw, state))
    va = add_reliability_features(apply_common_preprocess(va_raw, state))

    tr = add_temporal_platoon_features(tr, k=PLATOON_K)
    va = apply_platoon_lookup(
        va,
        fit_platoon_lookup(tr, k=PLATOON_K),
        k=PLATOON_K,
    )

    tr_contam = add_temporal_futures_features(tr_raw)
    for c in CONTAM_FEATURES:
        tr[c] = tr_contam[c].values

    va_contam = apply_futures_lookup(va_raw, fit_futures_lookup(tr_raw))
    for c in CONTAM_FEATURES:
        va[c] = va_contam[c].values

    return tr, va, tr_raw, va_raw

## 3. CatBoost 설정

In [9]:
from catboost import CatBoostClassifier, CatBoostRegressor

CAT_PARAMS = dict(
    loss_function="Logloss",
    eval_metric="BrierScore",
    iterations=2500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=10.0,
    random_seed=SEED,
    random_strength=0.5,
    border_count=128,
    verbose=False,
    allow_writing_files=False,
)

RESIDUAL_PARAMS = dict(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=2500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=10.0,
    random_seed=SEED,
    random_strength=0.5,
    border_count=128,
    verbose=False,
    allow_writing_files=False,
)

BASELINE_PARAMS = dict(
    loss_function="Logloss",
    eval_metric="BrierScore",
    iterations=700,
    learning_rate=0.04,
    depth=3,
    l2_leaf_reg=15.0,
    random_seed=SEED,
    random_strength=0.3,
    border_count=64,
    verbose=False,
    allow_writing_files=False,
)

# V3-A. Sample weighting

R/F hard split은 버리고 global 모델을 유지합니다.

가중치는 validation year에 하드코딩하지 않고,
**현재 training set의 최신 시즌에서 몇 시즌 전인지(age)** 로 계산합니다.
따라서 최종 2019~2024 학습에도 같은 recipe를 적용할 수 있습니다.

In [10]:
WEIGHT_RECIPES = {
    "equal": {"kind":"equal"},

    "global_recent": {
        "kind":"global_recent",
        "age_weight":{0:1.00, 1:0.85, 2:0.70, 3:0.60},
        "older":0.50,
    },

    "f_recent_moderate": {
        "kind":"f_recent",
        "age_weight":{0:1.00, 1:0.80, 2:0.60, 3:0.50},
        "older":0.50,
    },

    "f_recent_strong": {
        "kind":"f_recent",
        "age_weight":{0:1.00, 1:0.60, 2:0.40, 3:0.30},
        "older":0.30,
    },

    "f_recent_low_n": {
        "kind":"f_recent_low_n",
        "age_weight":{0:1.00, 1:0.80, 2:0.60, 3:0.50},
        "older":0.50,
        "low_n_floor":0.60,
        "low_n_k":200.0,
    },
}

def make_sample_weight(tr_raw, recipe_name):
    recipe = WEIGHT_RECIPES[recipe_name]
    w = pd.Series(np.ones(len(tr_raw)), index=tr_raw.index, dtype=float)

    if recipe["kind"] == "equal":
        return w

    latest = int(tr_raw["season"].max())
    age = latest - tr_raw["season"].astype(int)
    recency = age.map(
        lambda a: recipe["age_weight"].get(int(a), recipe["older"])
    ).astype(float)

    if recipe["kind"] == "global_recent":
        w *= recency

    else:
        is_f = tr_raw["game_type"].astype(str).eq("F")
        w.loc[is_f] *= recency.loc[is_f]

        if recipe["kind"] == "f_recent_low_n":
            if "asof_pitcher_n" in tr_raw.columns:
                n = pd.to_numeric(
                    tr_raw["asof_pitcher_n"], errors="coerce"
                ).fillna(0).clip(lower=0)
            else:
                n = pd.Series(0.0, index=tr_raw.index)

            k = recipe["low_n_k"]
            floor = recipe["low_n_floor"]
            rel = n / (n + k)
            w *= floor + (1 - floor) * rel

    return w.clip(0.05, 1.0)

for name in WEIGHT_RECIPES:
    demo = train.loc[train["season"] < 2024]
    w = make_sample_weight(demo, name)
    print(name, "mean/min/max =", round(w.mean(),4), round(w.min(),4), round(w.max(),4))

equal mean/min/max = 1.0 1.0 1.0
global_recent mean/min/max = 0.7318 0.5 1.0
f_recent_moderate mean/min/max = 0.9665 0.5 1.0
f_recent_strong mean/min/max = 0.9493 0.3 1.0
f_recent_low_n mean/min/max = 0.8949 0.3 0.9942


In [11]:
def fit_weighted_direct(tr_df, va_df, tr_raw, recipe_name, seed=42):
    features = get_model_features(tr_df)
    cat_idx, cat_names = get_cat_feature_indices(features)

    w = make_sample_weight(tr_raw, recipe_name)
    weight_map = pd.Series(w.values, index=tr_raw[ID_COL].values)
    sample_weight = tr_df[ID_COL].map(weight_map).fillna(1.0).values

    params = dict(CAT_PARAMS)
    params["random_seed"] = seed

    model = CatBoostClassifier(**params)
    model.fit(
        tr_df[features],
        tr_df[TARGET].astype(int),
        cat_features=cat_idx,
        sample_weight=sample_weight,
        eval_set=(va_df[features], va_df[TARGET].astype(int)),
        early_stopping_rounds=200,
        use_best_model=True,
        verbose=False,
    )

    pred = model.predict_proba(va_df[features])[:,1]

    return {
        "model":model,
        "features":features,
        "pred":pred,
        "metric":competition_like_score(va_df[TARGET], pred),
        "best_iteration":model.get_best_iteration(),
        "recipe":recipe_name,
        "seed":seed,
    }

## V3-A1. 2024 single-seed screening

여기는 bagging이 아닙니다.  
5개 recipe를 seed 42 하나로 빠르게 비교합니다.

In [12]:
v3_tr24, v3_va24, v3_tr24_raw, v3_va24_raw = build_submitted_fold(train, 2024)

rows = []
v3_weight_screen_pack = {}

for recipe in WEIGHT_RECIPES:
    print("\nrecipe:", recipe)
    pack = fit_weighted_direct(
        v3_tr24, v3_va24, v3_tr24_raw,
        recipe_name=recipe,
        seed=SCREEN_SEED,
    )
    v3_weight_screen_pack[recipe] = pack
    rows.append({
        "recipe":recipe,
        "seed":SCREEN_SEED,
        "best_iteration":pack["best_iteration"],
        **pack["metric"],
    })

v3_weight_screen_2024 = pd.DataFrame(rows).sort_values(
    "score_like", ascending=False
).reset_index(drop=True)

equal_score_24 = float(
    v3_weight_screen_2024.query("recipe == 'equal'")["score_like"].iloc[0]
)
v3_weight_screen_2024["delta_vs_equal"] = (
    v3_weight_screen_2024["score_like"] - equal_score_24
)

display(v3_weight_screen_2024)


recipe: equal

recipe: global_recent

recipe: f_recent_moderate

recipe: f_recent_strong

recipe: f_recent_low_n


,recipe,seed,best_iteration,brier,score_like,pred_mean,actual_rate,mean_gap,pred_std,delta_vs_equal
0,f_recent_strong,42,507,0.247767,816.568361,0.493297,0.486105,0.007192,0.043683,28.009634
1,f_recent_moderate,42,392,0.247818,796.169144,0.493463,0.486105,0.007358,0.041561,7.610417
2,global_recent,42,375,0.247823,794.378384,0.493438,0.486105,0.007333,0.042563,5.819657
3,f_recent_low_n,42,288,0.247831,791.079192,0.492990,0.486105,0.006885,0.040693,2.520465
4,equal,42,312,0.247837,788.558727,0.494032,0.486105,0.007927,0.041144,0.000000


## V3-A2. 2023 stress test — 2024 상위 2개 + equal만 확인

In [13]:
recipes_23 = [
    "equal",
    "f_recent_strong",
]

v3_tr23, v3_va23, v3_tr23_raw, v3_va23_raw = build_submitted_fold(train, 2023)

rows23 = []
for recipe in recipes_23:
    print("\n2023:", recipe)
    pack = fit_weighted_direct(
        v3_tr23, v3_va23, v3_tr23_raw,
        recipe_name=recipe,
        seed=SCREEN_SEED,
    )
    rows23.append({
        "recipe":recipe,
        "best_iteration":pack["best_iteration"],
        **pack["metric"],
    })
    del pack["model"]
    gc.collect()

v3_weight_stress_2023 = pd.DataFrame(rows23).sort_values("brier").reset_index(drop=True)
equal_brier_23 = float(
    v3_weight_stress_2023.query("recipe == 'equal'")["brier"].iloc[0]
)
v3_weight_stress_2023["brier_delta_vs_equal"] = (
    v3_weight_stress_2023["brier"] - equal_brier_23
)

display(v3_weight_stress_2023)


2023: equal

2023: f_recent_strong


,recipe,best_iteration,brier,score_like,pred_mean,actual_rate,mean_gap,pred_std,brier_delta_vs_equal
0,equal,2,0.249974,10.550042,0.502166,0.499957,0.002209,0.005870,0.000000
1,f_recent_strong,0,0.249984,6.344987,0.500691,0.499957,0.000734,0.002178,0.000011


## V3-A3. Weighting 승자

자동으로 2024 최고 recipe를 선택합니다.  
필요하면 아래 `WEIGHT_WINNER` 문자열만 직접 바꾸면 됩니다.

In [14]:
WEIGHT_WINNER = str(v3_weight_screen_2024.iloc[0]["recipe"])

print("WEIGHT_WINNER:", WEIGHT_WINNER)

r24 = v3_weight_screen_2024.query("recipe == @WEIGHT_WINNER").iloc[0]
print("2024 delta vs equal:", f'{r24["delta_vs_equal"]:+.4f}')

if WEIGHT_WINNER in set(v3_weight_stress_2023["recipe"]):
    r23 = v3_weight_stress_2023.query("recipe == @WEIGHT_WINNER").iloc[0]
    print("2023 brier delta vs equal:", f'{r23["brier_delta_vs_equal"]:+.8f}')

print("작은 +3~5점은 noise 후보로 봅니다.")

WEIGHT_WINNER: f_recent_strong
2024 delta vs equal: +28.0096
2023 brier delta vs equal: +0.00001051
작은 +3~5점은 noise 후보로 봅니다.


## V3-A4. Weighting 승자만 5-seed bagging

In [15]:
def run_direct_5seed(tr_df, va_df, tr_raw, recipe_name):
    seed_rows, preds, best_iters = [], [], []

    for seed in EXPERIMENT_SEEDS:
        print("direct", recipe_name, "seed", seed)
        pack = fit_weighted_direct(
            tr_df, va_df, tr_raw,
            recipe_name=recipe_name,
            seed=seed,
        )
        preds.append(pack["pred"])
        best_iters.append(pack["best_iteration"])
        seed_rows.append({
            "seed":seed,
            "best_iteration":pack["best_iteration"],
            **pack["metric"],
        })
        del pack["model"]
        gc.collect()

    bag_pred = np.column_stack(preds).mean(axis=1)
    metric = competition_like_score(va_df[TARGET], bag_pred)

    pred_df = pd.DataFrame({
        ID_COL:va_df[ID_COL].values,
        "y_true":va_df[TARGET].values,
        "pred":bag_pred,
    })

    valid_iters = [x for x in best_iters if x is not None and x >= 0]

    return {
        "seed_df":pd.DataFrame(seed_rows),
        "pred_df":pred_df,
        "bag_metric":metric,
        "median_iteration":int(np.median(valid_iters)) + 1,
    }

v3_weight_5seed = run_direct_5seed(
    v3_tr24, v3_va24, v3_tr24_raw, WEIGHT_WINNER
)

display(v3_weight_5seed["seed_df"])

print("\n=== WEIGHTED DIRECT 5-SEED ===")
for k,v in v3_weight_5seed["bag_metric"].items():
    print(f"{k:>12}: {v:.6f}")

print(
    "delta vs old 801.146:",
    f'{v3_weight_5seed["bag_metric"]["score_like"] - REFERENCE_VAL2024_5SEED:+.4f}'
)

display(evaluate_by_game_type(v3_weight_5seed["pred_df"], train))

v3_weight_5seed["pred_df"].to_csv(
    "val2024_v3_weighted_direct_5seed.csv",
    index=False,
)

direct f_recent_strong seed 11
direct f_recent_strong seed 22
direct f_recent_strong seed 33
direct f_recent_strong seed 44
direct f_recent_strong seed 55


,seed,best_iteration,brier,score_like,pred_mean,actual_rate,mean_gap,pred_std
0,11,410,0.247836,788.834224,0.493699,0.486105,0.007594,0.042643
1,22,354,0.247824,793.929787,0.493084,0.486105,0.006979,0.041780
2,33,348,0.247795,805.355998,0.492951,0.486105,0.006846,0.042037
3,44,330,0.247806,800.938186,0.493072,0.486105,0.006967,0.041457
4,55,279,0.247811,799.046885,0.493335,0.486105,0.007230,0.041445



=== WEIGHTED DIRECT 5-SEED ===
       brier: 0.247800
  score_like: 803.555914
   pred_mean: 0.493228
 actual_rate: 0.486105
    mean_gap: 0.007123
    pred_std: 0.041698
delta vs old 801.146: +2.4098


,subset,n,brier,score_like,pred_mean,actual_rate,mean_gap,pred_std
0,ALL,253507,0.247800,803.555914,0.493228,0.486105,0.007123,0.041698
1,R,223497,0.247891,801.604704,0.496834,0.489707,0.007127,0.041856
2,F,30010,0.247120,492.166301,0.466375,0.459280,0.007095,0.028689


# V3-B. Baseline + residual

구조:

```text
작고 보수적인 CatBoost baseline
→ 시간순 honest OOF baseline prediction
→ residual = y - p_baseline
→ CatBoostRegressor가 residual 학습
→ final p = clip(p_baseline + residual_pred)
```

Residual training row의 baseline은 같은 row의 label을 사용한 in-sample prediction이 아닙니다.

이 V3에서는 먼저 **probability residual**을 확인합니다.
유효할 때만 후속 버전에서 logit residual을 비교합니다.

In [16]:
BASELINE_FEATURE_CANDIDATES = [
    "balls", "strikes", "inning", "outs_when_up",
    "score_diff", "leverage_index", "win_expectancy",
    "game_type", "top_bottom", "base_state",
    "pitcher_hand", "batter_hand",
    "asof_pitcher_success_rate", "asof_batter_success_rate",
    "asof_pitcher_n", "asof_batter_n",
    "platoon_split_eb", "platoon_n_reliability",
    "fe_pitcher_futures_share", "fe_batter_futures_share",
    "fe_pitcher_prior_n_log",
]

def get_baseline_features(df):
    return [c for c in BASELINE_FEATURE_CANDIDATES if c in df.columns]

def fit_small_baseline(tr_df, va_df, tr_raw, recipe_name, seed=42):
    features = get_baseline_features(tr_df)
    cat_names = [c for c in CATEGORICAL_CANDIDATES if c in features]
    cat_idx = [features.index(c) for c in cat_names]

    w = make_sample_weight(tr_raw, recipe_name)
    weight_map = pd.Series(w.values, index=tr_raw[ID_COL].values)
    sample_weight = tr_df[ID_COL].map(weight_map).fillna(1.0).values

    params = dict(BASELINE_PARAMS)
    params["random_seed"] = seed

    model = CatBoostClassifier(**params)
    model.fit(
        tr_df[features],
        tr_df[TARGET].astype(int),
        cat_features=cat_idx,
        sample_weight=sample_weight,
        eval_set=(va_df[features], va_df[TARGET].astype(int)),
        early_stopping_rounds=100,
        use_best_model=True,
        verbose=False,
    )

    return model, features, model.predict_proba(va_df[features])[:,1]

In [17]:
def build_honest_baseline_oof(full_train, train_end_year, recipe_name, seed=42):
    # residual training용 honest time OOF baseline prediction
    target_raw = full_train.loc[full_train["season"] < train_end_year].copy()
    seasons = sorted(target_raw["season"].dropna().unique())
    chunks = []

    for season in seasons:
        current = target_raw.loc[target_raw["season"] == season].copy()
        history = target_raw.loc[target_raw["season"] < season]

        if len(history) == 0:
            pred = np.full(len(current), 0.5, dtype=float)
        else:
            tr_fold, va_fold, tr_fold_raw, _ = build_submitted_fold(
                target_raw, int(season)
            )
            base_model, _, pred = fit_small_baseline(
                tr_fold, va_fold, tr_fold_raw,
                recipe_name=recipe_name,
                seed=seed,
            )
            del base_model
            gc.collect()

        chunks.append(pd.DataFrame({
            ID_COL:current[ID_COL].values,
            "y_true":current[TARGET].values,
            "season":current["season"].values,
            "base_pred":pred,
        }))

    return pd.concat(chunks, ignore_index=True)

In [18]:
def fit_residual_model(
    tr_df, va_df, tr_raw,
    oof_base_df, va_base_pred,
    recipe_name, seed=42
):
    features = get_model_features(tr_df)
    cat_idx, cat_names = get_cat_feature_indices(features)

    base_map = pd.Series(
        oof_base_df["base_pred"].values,
        index=oof_base_df[ID_COL].values,
    )
    base_tr = tr_df[ID_COL].map(base_map).astype(float)

    if base_tr.isna().any():
        raise ValueError("residual train baseline prediction missing")

    y_resid = tr_df[TARGET].astype(float).values - base_tr.values

    w = make_sample_weight(tr_raw, recipe_name)
    weight_map = pd.Series(w.values, index=tr_raw[ID_COL].values)
    sample_weight = tr_df[ID_COL].map(weight_map).fillna(1.0).values

    params = dict(RESIDUAL_PARAMS)
    params["random_seed"] = seed

    model = CatBoostRegressor(**params)
    va_resid = va_df[TARGET].astype(float).values - np.asarray(va_base_pred)

    model.fit(
        tr_df[features],
        y_resid,
        cat_features=cat_idx,
        sample_weight=sample_weight,
        eval_set=(va_df[features], va_resid),
        early_stopping_rounds=200,
        use_best_model=True,
        verbose=False,
    )

    resid_pred = model.predict(va_df[features])
    pred = np.clip(np.asarray(va_base_pred) + resid_pred, 1e-6, 1-1e-6)

    return {
        "model":model,
        "pred":pred,
        "metric":competition_like_score(va_df[TARGET], pred),
        "best_iteration":model.get_best_iteration(),
    }

## V3-B1. 2024 baseline + residual single-seed

In [19]:
print("honest OOF baseline 생성 중...")

v3_base_oof_24 = build_honest_baseline_oof(
    train, 2024, WEIGHT_WINNER, SCREEN_SEED
)

v3_base24_model, v3_base24_features, v3_base24_pred = fit_small_baseline(
    v3_tr24, v3_va24, v3_tr24_raw,
    WEIGHT_WINNER, SCREEN_SEED
)

print("\n=== BASELINE ONLY ===")
for k,v in competition_like_score(v3_va24[TARGET], v3_base24_pred).items():
    print(f"{k:>12}: {v:.6f}")

v3_residual24_single = fit_residual_model(
    v3_tr24, v3_va24, v3_tr24_raw,
    v3_base_oof_24, v3_base24_pred,
    WEIGHT_WINNER, SCREEN_SEED
)

print("\n=== BASELINE + RESIDUAL ===")
for k,v in v3_residual24_single["metric"].items():
    print(f"{k:>12}: {v:.6f}")

weighted_single = float(
    v3_weight_screen_2024.query("recipe == @WEIGHT_WINNER")["score_like"].iloc[0]
)
print(
    "delta vs weighted-direct single:",
    f'{v3_residual24_single["metric"]["score_like"] - weighted_single:+.4f}'
)

honest OOF baseline 생성 중...

=== BASELINE ONLY ===
       brier: 0.249054
  score_like: 301.546878
   pred_mean: 0.500496
 actual_rate: 0.486105
    mean_gap: 0.014391
    pred_std: 0.043597

=== BASELINE + RESIDUAL ===
       brier: 0.248688
  score_like: 448.057012
   pred_mean: 0.497069
 actual_rate: 0.486105
    mean_gap: 0.010964
    pred_std: 0.059110
delta vs weighted-direct single: -368.5113


## V3-B2. 2023 residual stress test

In [20]:
v3_base_oof_23 = build_honest_baseline_oof(
    train, 2023, WEIGHT_WINNER, SCREEN_SEED
)

v3_base23_model, _, v3_base23_pred = fit_small_baseline(
    v3_tr23, v3_va23, v3_tr23_raw,
    WEIGHT_WINNER, SCREEN_SEED
)

v3_residual23_single = fit_residual_model(
    v3_tr23, v3_va23, v3_tr23_raw,
    v3_base_oof_23, v3_base23_pred,
    WEIGHT_WINNER, SCREEN_SEED
)

print("=== RESIDUAL 2023 ===")
for k,v in v3_residual23_single["metric"].items():
    print(f"{k:>12}: {v:.6f}")

display(
    evaluate_by_game_type(
        pd.DataFrame({
            ID_COL:v3_va23[ID_COL].values,
            "y_true":v3_va23[TARGET].values,
            "pred":v3_residual23_single["pred"],
        }),
        train
    )
)

=== RESIDUAL 2023 ===
       brier: 0.250179
  score_like: 0.000000
   pred_mean: 0.504556
 actual_rate: 0.499957
    mean_gap: 0.004599
    pred_std: 0.020774


,subset,n,brier,score_like,pred_mean,actual_rate,mean_gap,pred_std
0,ALL,245525,0.250179,0.000000,0.504556,0.499957,0.004599,0.020774
1,R,219839,0.249567,169.252874,0.498939,0.503118,-0.004180,0.012598
2,F,25686,0.255416,0.000000,0.552637,0.472904,0.079734,0.013597


## V3-B3. Residual 5-seed

Single-seed에서 residual이 명확히 실패했다면
`RUN_RESIDUAL_5SEED=False`로 두고 건너뛰세요.

실험 bagging은 여기서도 **5 seeds만** 사용합니다.

In [21]:
RUN_RESIDUAL_5SEED = True
v3_residual_5seed = None

if RUN_RESIDUAL_5SEED:
    seed_rows, preds, best_iters = [], [], []

    for seed in EXPERIMENT_SEEDS:
        print("residual seed", seed)
        pack = fit_residual_model(
            v3_tr24, v3_va24, v3_tr24_raw,
            v3_base_oof_24, v3_base24_pred,
            WEIGHT_WINNER, seed
        )
        preds.append(pack["pred"])
        best_iters.append(pack["best_iteration"])
        seed_rows.append({
            "seed":seed,
            "best_iteration":pack["best_iteration"],
            **pack["metric"],
        })
        del pack["model"]
        gc.collect()

    bag_pred = np.column_stack(preds).mean(axis=1)
    bag_metric = competition_like_score(v3_va24[TARGET], bag_pred)
    valid_iters = [x for x in best_iters if x is not None and x >= 0]

    v3_residual_5seed = {
        "seed_df":pd.DataFrame(seed_rows),
        "pred_df":pd.DataFrame({
            ID_COL:v3_va24[ID_COL].values,
            "y_true":v3_va24[TARGET].values,
            "pred":bag_pred,
        }),
        "bag_metric":bag_metric,
        "median_iteration":int(np.median(valid_iters)) + 1,
    }

    display(v3_residual_5seed["seed_df"])

    print("\n=== RESIDUAL 5-SEED ===")
    for k,v in bag_metric.items():
        print(f"{k:>12}: {v:.6f}")

    print(
        "delta vs old 801.146:",
        f'{bag_metric["score_like"] - REFERENCE_VAL2024_5SEED:+.4f}'
    )

    display(evaluate_by_game_type(v3_residual_5seed["pred_df"], train))

    v3_residual_5seed["pred_df"].to_csv(
        "val2024_v3_residual_5seed.csv",
        index=False,
    )
else:
    print("Residual 5-seed skipped.")

residual seed 11
residual seed 22
residual seed 33
residual seed 44
residual seed 55


,seed,best_iteration,brier,score_like,pred_mean,actual_rate,mean_gap,pred_std
0,11,68,0.248684,449.370363,0.496836,0.486105,0.010731,0.059953
1,22,74,0.248692,446.151641,0.496309,0.486105,0.010205,0.060792
2,33,77,0.248683,449.879342,0.496379,0.486105,0.010274,0.060640
3,44,74,0.248688,447.733625,0.496617,0.486105,0.010512,0.060318
4,55,67,0.248674,453.421483,0.496985,0.486105,0.010880,0.059458



=== RESIDUAL 5-SEED ===
       brier: 0.248681
  score_like: 450.893091
   pred_mean: 0.496625
 actual_rate: 0.486105
    mean_gap: 0.010520
    pred_std: 0.060202
delta vs old 801.146: -350.2530


,subset,n,brier,score_like,pred_mean,actual_rate,mean_gap,pred_std
0,ALL,253507,0.248681,450.893091,0.496625,0.486105,0.010520,0.060202
1,R,223497,0.248423,588.795043,0.494564,0.489707,0.004857,0.060639
2,F,30010,0.250601,0.000000,0.511979,0.459280,0.052699,0.054438


# V3 결과 한눈에 비교

In [22]:
rows = [
    {
        "model":"OLD_M2_CONTAM_5SEED_REFERENCE",
        "score_like":REFERENCE_VAL2024_5SEED,
        "brier":np.nan,
        "mean_gap":np.nan,
    },
    {
        "model":f"WEIGHTED_DIRECT::{WEIGHT_WINNER}",
        "score_like":v3_weight_5seed["bag_metric"]["score_like"],
        "brier":v3_weight_5seed["bag_metric"]["brier"],
        "mean_gap":v3_weight_5seed["bag_metric"]["mean_gap"],
    },
]

if v3_residual_5seed is not None:
    rows.append({
        "model":f"BASELINE_RESIDUAL::{WEIGHT_WINNER}",
        "score_like":v3_residual_5seed["bag_metric"]["score_like"],
        "brier":v3_residual_5seed["bag_metric"]["brier"],
        "mean_gap":v3_residual_5seed["bag_metric"]["mean_gap"],
    })

v3_final_compare = pd.DataFrame(rows).sort_values(
    "score_like", ascending=False
).reset_index(drop=True)

display(v3_final_compare)

,model,score_like,brier,mean_gap
0,WEIGHTED_DIRECT::f_recent_strong,803.555914,0.247800,0.007123
1,OLD_M2_CONTAM_5SEED_REFERENCE,801.146099,NaN,NaN
2,BASELINE_RESIDUAL::f_recent_strong,450.893091,0.248681,0.010520


# 최종 제출용 10-seed

여기부터는 **실험이 아니라 제출 직전용**입니다.

기본값은 `RUN_FINAL_10SEED=False`.

V3 결과를 보고 최종 구조가 확정된 뒤에만:
- `FINAL_MODEL_KIND`
- `FINAL_WEIGHT_RECIPE`

를 고정하고 `True`로 변경하세요.

실험 중에는 10-seed를 반복하지 않습니다.

In [23]:
# RUN_FINAL_10SEED = False

# FINAL_MODEL_KIND = "WEIGHTED_DIRECT"
# # FINAL_MODEL_KIND = "BASELINE_RESIDUAL"

# FINAL_WEIGHT_RECIPE = WEIGHT_WINNER

# print("RUN_FINAL_10SEED:", RUN_FINAL_10SEED)
# print("FINAL_MODEL_KIND:", FINAL_MODEL_KIND)
# print("FINAL_WEIGHT_RECIPE:", FINAL_WEIGHT_RECIPE)
# print("FINAL_SEEDS:", FINAL_SEEDS)

In [24]:
# def build_full_train_test(full_train, test_df):
#     state = fit_preprocess_state(full_train)

#     tr = add_reliability_features(
#         apply_common_preprocess(full_train.copy(), state)
#     )
#     te = add_reliability_features(
#         apply_common_preprocess(test_df.copy(), state)
#     )

#     tr = add_temporal_platoon_features(tr, k=PLATOON_K)
#     platoon_lookup = fit_platoon_lookup(tr, k=PLATOON_K)
#     te = apply_platoon_lookup(te, platoon_lookup, k=PLATOON_K)

#     tr_contam = add_temporal_futures_features(full_train)
#     for c in CONTAM_FEATURES:
#         tr[c] = tr_contam[c].values

#     contam_lookup = fit_futures_lookup(full_train)
#     te_contam = apply_futures_lookup(test_df, contam_lookup)
#     for c in CONTAM_FEATURES:
#         te[c] = te_contam[c].values

#     return tr, te

In [25]:
# if RUN_FINAL_10SEED:
#     final_tr, final_te = build_full_train_test(train, test)

#     final_features = get_model_features(final_tr)
#     final_cat_idx, _ = get_cat_feature_indices(final_features)

#     w = make_sample_weight(train, FINAL_WEIGHT_RECIPE)
#     w_map = pd.Series(w.values, index=train[ID_COL].values)
#     final_sample_weight = final_tr[ID_COL].map(w_map).fillna(1.0).values

#     final_pred_list = []
#     log_rows = []

#     if FINAL_MODEL_KIND == "WEIGHTED_DIRECT":
#         final_iterations = v3_weight_5seed["median_iteration"]

#         for seed in FINAL_SEEDS:
#             t0 = time.time()
#             params = dict(CAT_PARAMS)
#             params["random_seed"] = seed
#             params["iterations"] = final_iterations

#             model = CatBoostClassifier(**params)
#             model.fit(
#                 final_tr[final_features],
#                 final_tr[TARGET].astype(int),
#                 cat_features=final_cat_idx,
#                 sample_weight=final_sample_weight,
#                 verbose=False,
#             )
#             final_pred_list.append(
#                 model.predict_proba(final_te[final_features])[:,1]
#             )
#             log_rows.append({"seed":seed, "minutes":(time.time()-t0)/60})
#             del model
#             gc.collect()

#     elif FINAL_MODEL_KIND == "BASELINE_RESIDUAL":
#         if v3_residual_5seed is None:
#             raise RuntimeError("Residual 5-seed를 먼저 실행하세요.")

#         final_base_oof = build_honest_baseline_oof(
#             train,
#             int(train["season"].max()) + 1,
#             FINAL_WEIGHT_RECIPE,
#             SCREEN_SEED,
#         )

#         base_features = get_baseline_features(final_tr)
#         base_cat_names = [c for c in CATEGORICAL_CANDIDATES if c in base_features]
#         base_cat_idx = [base_features.index(c) for c in base_cat_names]

#         base_model = CatBoostClassifier(**BASELINE_PARAMS)
#         base_model.fit(
#             final_tr[base_features],
#             final_tr[TARGET].astype(int),
#             cat_features=base_cat_idx,
#             sample_weight=final_sample_weight,
#             verbose=False,
#         )
#         base_test_pred = base_model.predict_proba(final_te[base_features])[:,1]

#         base_map = pd.Series(
#             final_base_oof["base_pred"].values,
#             index=final_base_oof[ID_COL].values,
#         )
#         base_train_pred = final_tr[ID_COL].map(base_map).astype(float)

#         if base_train_pred.isna().any():
#             raise RuntimeError("final residual OOF baseline missing")

#         y_resid = final_tr[TARGET].astype(float).values - base_train_pred.values
#         final_iterations = v3_residual_5seed["median_iteration"]

#         for seed in FINAL_SEEDS:
#             t0 = time.time()
#             params = dict(RESIDUAL_PARAMS)
#             params["random_seed"] = seed
#             params["iterations"] = final_iterations

#             model = CatBoostRegressor(**params)
#             model.fit(
#                 final_tr[final_features],
#                 y_resid,
#                 cat_features=final_cat_idx,
#                 sample_weight=final_sample_weight,
#                 verbose=False,
#             )
#             resid_pred = model.predict(final_te[final_features])
#             final_pred_list.append(
#                 np.clip(base_test_pred + resid_pred, 1e-6, 1-1e-6)
#             )
#             log_rows.append({"seed":seed, "minutes":(time.time()-t0)/60})
#             del model
#             gc.collect()

#     else:
#         raise ValueError(FINAL_MODEL_KIND)

#     final_test_pred = np.column_stack(final_pred_list).mean(axis=1)
#     final_test_pred = np.clip(final_test_pred, 1e-6, 1-1e-6)

#     display(pd.DataFrame(log_rows))
#     print("pred mean/std:", final_test_pred.mean(), final_test_pred.std())

# else:
#     print("Final 10-seed skipped. V3 결과 확정 후에만 실행하세요.")

In [26]:
# if RUN_FINAL_10SEED:
#     submission = test[[ID_COL]].copy()
#     submission[TARGET] = final_test_pred

#     assert submission[TARGET].between(0,1).all()
#     assert submission[TARGET].notna().all()

#     path = Path(f"./submission_v3_{FINAL_MODEL_KIND.lower()}_10seed.csv")
#     submission.to_csv(path, index=False)

#     display(submission.head())
#     print("saved:", path.resolve())
# else:
#     print("No final submission created.")

# 판단 기준

### Weighting
- +3~5: noise 가능성 큼
- +10 이상: 후보
- +20 이상: 강한 후보
- 2023에서 완전히 반대로 가면 경계

### Residual
다음 네 가지를 같이 봅니다.
- weighted direct 대비 Brier
- 2024 mean_gap
- R/F
- 2023 stress 방향

### 핵심
2024 숫자만 억지로 900으로 만드는 것이 아니라,
**시간순 검증과 행 독립성을 유지하면서 2025에 전이될 수 있는 구조**를 찾습니다.